In [ ]:
# ArXiv contains research papers
# ArXiv API client for Python
# This will help us to fetch papers from arXiv

# Wikipedia is a Python library that makes it easy to access and parse data from Wikipedia.

# !pip install arxiv wikipedia langchainhub

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY", "")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ.setdefault("USER_AGENT", "langchain-samples/1.0")

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
wiki.name

'wikipedia'

In [ ]:
# This code block is for loading documents from the web, splitting them into chunks, and creating a vector store for retrieval.
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vectordb.as_retriever()
# retriever


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# This code block is for creating a retriever tool that can be used to search for information about LangSmith. 
# The tool will use the retriever we created in the previous code block to fetch relevant documents from the vector store.
from langchain_core.tools.retriever import create_retriever_tool

# "retriever_tool.name"
retriever_tool = create_retriever_tool(retriever, "langsmith_search",
                      "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

In [ ]:
retriever_tool.name

'langsmith_search'

In [ ]:
## Arxiv Tool
# This code block is for creating a tool that can be used to search for research papers on arXiv.

from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [ ]:
# Combine all tools 
tools=[wiki,arxiv,retriever_tool]

tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\vivek\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 Tool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000002D087A7B880>, retriever=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vecto

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# In LangChain v1, prompts are passed directly to create_agent via system_prompt
# No need for ChatPromptTemplate with agent_scratchpad

In [ ]:
### Agents
# LangChain v1 uses create_agent (replaces create_openai_tools_agent + AgentExecutor)
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
)

In [ ]:
# In LangChain v1, create_agent returns a compiled StateGraph directly (no AgentExecutor needed)
agent

In [ ]:
# LangChain v1: invoke using messages format
response = agent.invoke({"messages": [{"role": "user", "content": "Tell me about Langsmith"}]})

# Print the final response
print(response["messages"][-1].content)